# Đánh giá SVM tốt nhất

GridSearchCV thử 16 tổ hợp với Stratified CV 5 fold và chọn theo F1-macro.
Pipeline gồm StandardScaler và SVC kernel RBF; scaler được fit riêng trên train của từng fold. Tập test chỉ dùng để đánh giá cuối cùng.

Lưới tham số: `svc__C = [0.1, 1, 10, 100]`, `svc__gamma = ["scale", 0.01, 0.1, 1]`.


## 1. Thư viện và đường dẫn

In [1]:
import json
from pathlib import Path
import sys

import joblib
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

project_dir = Path.cwd().resolve()
if project_dir.name == "svm":
    project_dir = project_dir.parents[1]
elif project_dir.name == "models":
    project_dir = project_dir.parent
sys.path.insert(0, str(project_dir))

from models.common import read_split, timestamp, write_json
from models.reporting import SUMMARY_COLUMNS

model_dir = project_dir / "models" / "svm"
pd.set_option("display.max_columns", None)

## 2. Đọc model và dữ liệu

In [2]:
artifact = joblib.load(model_dir / "training_results.joblib")
search = artifact["search"]
model = joblib.load(model_dir / "model.joblib")
X_train, y_train = read_split("train")
X_test, y_test = read_split("test")
classes = model.classes_.tolist()
results = search.cv_results_
best_index = int(search.best_index_)

## 3. Siêu tham số tốt nhất

In [3]:
display(pd.DataFrame([search.best_params_]))

,svc__C,svc__gamma
0,1,1


## 4. Bảng tổng hợp

In [4]:
train_prediction = model.predict(X_train)
prediction = model.predict(X_test)

options = {"labels": classes, "average": "macro", "zero_division": 0}
summary = pd.DataFrame([{
    "Model": "SVM", "Configuration": "GridSearch",
    "CV_accuracy_mean": float(results["mean_test_accuracy"][best_index]),
    "CV_f1_macro_mean": float(results["mean_test_f1_macro"][best_index]),
    "CV_f1_macro_std": float(results["std_test_f1_macro"][best_index]),
    "Test_accuracy": float(accuracy_score(y_test, prediction)),
    "Test_precision_macro": float(precision_score(y_test, prediction, **options)),
    "Test_recall_macro": float(recall_score(y_test, prediction, **options)),
    "Test_f1_macro": float(f1_score(y_test, prediction, **options)),
    "Train_f1_macro": float(f1_score(y_train, train_prediction, **options)),
    "CV_train_validation_gap_F1": float(
        results["mean_train_f1_macro"][best_index] - results["mean_test_f1_macro"][best_index]
    ),
    "Preprocessing": "StandardScaler",
    "Best_grid_parameters": json.dumps(search.best_params_, sort_keys=True),
}], columns=SUMMARY_COLUMNS)
display(summary)

,Model,Configuration,CV_accuracy_mean,CV_f1_macro_mean,CV_f1_macro_std,Test_accuracy,Test_precision_macro,Test_recall_macro,Test_f1_macro,Train_f1_macro,CV_train_validation_gap_F1,Preprocessing,Best_grid_parameters
0,SVM,GridSearch,0.984659,0.984613,0.004663,0.990909,0.991539,0.990909,0.990895,0.992609,0.008709,StandardScaler,"{""svc__C"": 1, ""svc__gamma"": 1}"


## 5. Kết quả GridSearch

In [5]:
grid_rows = []
for i, parameters in enumerate(results["params"]):
    grid_rows.append({
        "Model": "SVM", "Candidate": i + 1,
        "Parameters": json.dumps(parameters, sort_keys=True),
        "Selected": i == best_index,
        "CV_accuracy_mean": float(results["mean_test_accuracy"][i]),
        "CV_f1_macro_mean": float(results["mean_test_f1_macro"][i]),
        "CV_f1_macro_std": float(results["std_test_f1_macro"][i]),
    })
grid = pd.DataFrame(grid_rows)
display(grid)

,Model,Candidate,Parameters,Selected,CV_accuracy_mean,CV_f1_macro_mean,CV_f1_macro_std
0,SVM,1,"{""svc__C"": 0.1, ""svc__gamma"": ""scale""}",False,0.935227,0.933384,0.006856
1,SVM,2,"{""svc__C"": 0.1, ""svc__gamma"": 0.01}",False,0.790341,0.768720,0.018489
2,SVM,3,"{""svc__C"": 0.1, ""svc__gamma"": 0.1}",False,0.910795,0.907106,0.004857
3,SVM,4,"{""svc__C"": 0.1, ""svc__gamma"": 1}",False,0.969318,0.969044,0.011588
4,SVM,5,"{""svc__C"": 1, ""svc__gamma"": ""scale""}",False,0.978977,0.978803,0.010624
5,SVM,6,"{""svc__C"": 1, ""svc__gamma"": 0.01}",False,0.927273,0.925644,0.005234
6,SVM,7,"{""svc__C"": 1, ""svc__gamma"": 0.1}",False,0.976136,0.976045,0.010135
7,SVM,8,"{""svc__C"": 1, ""svc__gamma"": 1}",True,0.984659,0.984613,0.004663
8,SVM,9,"{""svc__C"": 10, ""svc__gamma"": ""scale""}",False,0.982386,0.982212,0.004895
9,SVM,10,"{""svc__C"": 10, ""svc__gamma"": 0.01}",False,0.978409,0.978005,0.009650


## 6. Lưu kết quả đánh giá

In [7]:
evaluated_at = timestamp()
evaluation = {
    "model_name": "SVM", "evaluated_at": evaluated_at,
    "tables": {
        "Tong_hop": summary.to_dict(orient="records"),
        "GridSearch": grid.to_dict(orient="records"),
    },
}
selected = summary.iloc[0]
metadata = {
    "model_file": "model.joblib", "model_name": "SVM",
    "estimator": "SVC", "kernel": model.named_steps["svc"].kernel, "configuration": "GridSearch",
    "parameters": search.best_params_, "preprocessing": "StandardScaler",
    "input_format": "Unscaled feature values in the order listed in features",
    "features": X_train.columns.tolist(), "classes": classes,
    "selection_metric": "CV_f1_macro_mean",
    "cv_f1_macro_mean": float(selected["CV_f1_macro_mean"]),
    "cv_f1_macro_std": float(selected["CV_f1_macro_std"]),
    "train_samples": len(X_train), "test_samples": len(X_test),
    "n_splits": search.n_splits_, "cv_method": str(search.cv),
    "training_finished_at": artifact["finished_at"],
    "versions": artifact["versions"], "evaluated_at": evaluated_at,
    "test_metrics": {
        "accuracy": float(selected["Test_accuracy"]),
        "precision_macro": float(selected["Test_precision_macro"]),
        "recall_macro": float(selected["Test_recall_macro"]),
        "f1_macro": float(selected["Test_f1_macro"]),
    },
}
write_json(model_dir / "model_metadata.json", metadata)
write_json(model_dir / "evaluation_results.json", evaluation)